# Embeddings und relevante Suche – Konzepte

**Geschäftliches Problem:** Nützliche Stellenanzeigen sollen möglichst weit oben in einer nach Relevanz sortierten Ergebnisliste erscheinen.

Ein Modell unterstützt eine Entscheidung, trifft aber nicht die geschäftliche Entscheidung für uns.

## Weiterführende Übungen

- Kann die Zusammenführung von Ranglisten (Fusion) eine Stellenanzeige finden, die in beiden Kandidatenlisten fehlt?
- Warum kann ein hoher Kosinus-Ähnlichkeitswert trotzdem bedeuten, dass eine Anforderung an den Arbeitsort nicht erfüllt wird?
- Ist ein Klick eine unverzerrte Bewertung der Relevanz?

## Datensatz und praktischer Umfang

Die unten bereitgestellten Ranglisten wurden künstlich erstellt. Die Übung verwendet 24 freigegebene Textausschnitte und die dazugehörigen vorläufigen Relevanzbewertungen.

Der gesamte Code ist bereits vorgegeben. Führe die Codezellen von oben nach unten aus.

Die zugehörige Übung und Lösung bewerten zunächst 24 freigegebene Textausschnitte und demonstrieren anschliessend eine interaktive Suche über die 5'011 echten Stellenanzeigen aus Vorlesung 2. Für die grössere Sammlung liegen keine vollständigen Relevanzbewertungen der Stellenanzeigen vor. Deshalb werden ihre Suchergebnisse manuell untersucht und nicht anhand der Relevanzbewertungen der Textausschnitte bewertet.

## Bibliotheken und Einstellungen

Verwende den vorbereiteten Python-3.11-Kernel im Codespace. Es werden weder eine GPU noch ein API-Schlüssel benötigt.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 7
np.random.seed(SEED)
DATA_ROOT = Path('../../../00_Data')

## 1. Suchziel und Basismodell festlegen

Der Benutzer gibt eine Suchanfrage ein. Die Relevanz bezieht sich auf die Kombination aus Suchanfrage und Stellenanzeige, nicht auf eine dauerhaft festgelegte Kategorie.

BM25 dient als wortbasiertes Basismodell und berücksichtigt Worthäufigkeiten, die Seltenheit von Begriffen und die Textlänge.

Die genaue Übereinstimmung von Fähigkeiten kann wichtig sein. Gleiche Wörter bedeuten jedoch nicht automatisch, dass eine Stellenanzeige zur gesuchten Tätigkeit passt.

## 2. Für die Suche trainierte Embeddings

Die Stellenanzeigen werden einmal codiert und ihre Vektoren zwischengespeichert. Jede neue Suchanfrage wird ebenfalls in einen Vektor umgewandelt und mit den gespeicherten Vektoren verglichen.

Durch Pooling der Ausgaben eines gewöhnlichen BERT-Modells können Vektoren zwar zwischengespeichert werden, das Modell wurde dadurch aber nicht darauf trainiert, sinnvolle Ähnlichkeitsbeziehungen anhand der Kosinus-Ähnlichkeit abzubilden.

E5 ist ein bereits fertig trainierter, mehrsprachiger Encoder für die Informationssuche, der über die Bibliothek Sentence Transformers verwendet wird.

In [2]:
query_vector = np.array([1., 0.])
ad_vectors = np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.3]])
ad_vectors /= np.linalg.norm(ad_vectors, axis=1, keepdims=True)
display(pd.DataFrame({'ad': ['A', 'B', 'C'],
                      'constructed cosine': ad_vectors @ query_vector})
        .sort_values('constructed cosine', ascending=False))

,ad,constructed cosine
0,A,0.993884
2,C,0.919145
1,B,0.242536


## 3. Ranglisten kombinieren, nicht rohe Bewertungsskalen

Reciprocal Rank Fusion (RRF) kombiniert mehrere Ranglisten, indem für jedes Ergebnis die Werte 1 / (60 + Rang) über alle Listen aufsummiert werden.

Die Rangposition beginnt bei 1. Wenn ein Ergebnis in einer Liste fehlt, trägt diese Liste den Wert 0 bei.

Die Kombination verschiedener Bewertungssignale kann allerdings auch einen Fehler verstärken.

In [3]:
supplied_lists = [['A', 'B', 'C'], ['A', 'C', 'D']]
votes = {}
for ranking in supplied_lists:
    for rank, ad in enumerate(ranking, 1):
        votes[ad] = votes.get(ad, 0) + 1 / (60 + rank)
display(pd.Series(votes, name='RRF score').sort_values(ascending=False).to_frame())

,RRF score
A,0.032787
C,0.032002
B,0.016129
D,0.015873


## 4. Eine nützliche Top-3-Ergebnisliste bewerten

Angenommen, nur B und D sind relevant und die zurückgegebene Reihenfolge lautet A, B, C.

Precision misst den Anteil relevanter Ergebnisse unter den zurückgegebenen Treffern. Recall misst, welcher Anteil aller relevanten Ergebnisse gefunden wurde. Der Reciprocal Rank berücksichtigt die Position des ersten relevanten Ergebnisses.

Der Mean Reciprocal Rank bildet den Durchschnitt dieser Werte über mehrere Suchanfragen.

In [4]:
ranking, relevant, k = ['A', 'B', 'C'], {'B', 'D'}, 3
positions = [rank for rank, ad in enumerate(ranking[:k], 1) if ad in relevant]
display(pd.Series({'Precision@3': len(positions) / k,
                   'Recall@3': len(positions) / len(relevant),
                   'RR@3': 1 / positions[0] if positions else 0.}))

Precision@3    0.333333
Recall@3       0.500000
RR@3           0.500000
dtype: float64

## Interpretation

Die gleiche Position des ersten relevanten Treffers bedeutet nicht automatisch, dass gleich viele relevante Ergebnisse gefunden wurden.

Ähnlichkeitswerte sind keine Relevanzbewertungen.

Definiere vor der Bewertung die Suchabsicht und überprüfe Anforderungen wie den Arbeitsort oder die Arbeitsschichten unabhängig vom Ähnlichkeitswert.

## Zusammenfassung und geschäftliche Auswirkungen

Wähle einen Ansatz anhand der gemessenen Ergebnisse aus und nicht deshalb, weil er aus mehr Komponenten besteht.

Überprüfe den tatsächlichen Nutzen für die Benutzer, die Antwortzeit und die Kosten getrennt von den offline berechneten Kennzahlen der Informationssuche.

## Einschränkungen

Dies ist ein kleines Lehrbeispiel und kein Nachweis dafür, wie gut das System im praktischen Einsatz funktionieren würde.

Halte die Evaluation von der Anpassung und Auswahl der Modelle getrennt.

Untersuche auch Fehler und die Entscheidungen der Benutzer, statt dich nur auf eine einzelne Kennzahl zu konzentrieren.

## Weiterführende Übungen

- Kann die Kombination von Ranglisten eine Stellenanzeige finden, die in beiden Kandidatenlisten fehlt?
- Warum kann ein hoher Kosinus-Ähnlichkeitswert trotzdem eine Anforderung an den Arbeitsort nicht erfüllen?
- Ist ein Klick eine unverzerrte Bewertung der Relevanz?

## References

**Required reading:** https://www.sbert.net/examples/applications/semantic-search/README.html
and https://huggingface.co/intfloat/multilingual-e5-small
**Instructor references:** Reimers & Gurevych (2019), *Sentence-BERT*, https://aclanthology.org/D19-1410/;
Wang et al. (2024), *Multilingual E5 Text Embeddings*, https://arxiv.org/abs/2402.05672;
Cormack et al. (2009), *Reciprocal rank fusion*, https://doi.org/10.1145/1571941.1572114.
**Official BM25 documentation:** https://www.elastic.co/docs/reference/elasticsearch/index-settings/similarity

### Jupyter notebook - footer info

Include your name, submission date and a short description of your changes when submitting.